# 04 - Save and check the model

Fit the selected K-Means settings on training data, save the full pipeline, and check a reload. Test metrics are reported only after selection. This creates a local bundle; it does not upload or deploy it.

In [1]:
from pathlib import Path
import json
import platform
import sys

assert platform.system() == "Linux", "Use the WSL project kernel."
assert Path(sys.prefix).name == "mlop-pj", "Select the project environment."
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.bundle import save_bundle, predict_profiles
selection = json.loads((ROOT / "reports/selection.json").read_text())
assert selection["ready_for_bundle"], "Review Notebook 03 before continuing."
print("Selected:", selection["zero_policy"], "k =", selection["k"])


Selected: keep_zero k = 5


## 1. Fit and save

The bundle includes preprocessing, K-Means, feature order, segment IDs, and distance thresholds. Thresholds use training distance percentiles; distance is not a confidence probability. Do not tune settings using the test score.

In [2]:
cluster_profiles, manifest = save_bundle(ROOT)
cluster_profiles


,Category 1,Category 2,Category 3,Category 4,Category 5,Category 6,Category 7,Category 8,Category 9,Category 10,...,Category 15,Category 16,Category 17,Category 18,Category 19,Category 20,Category 21,Category 22,Category 23,Category 24
cluster_id,,,,,,,,,,,,,,,,,,,,,
0,1.162,2.230,1.985,2.212,2.445,3.048,4.251,3.526,4.453,3.831,...,2.420,0.967,0.639,0.515,0.517,0.639,0.709,1.074,0.988,1.100
1,0.722,1.020,1.662,1.636,1.710,1.736,2.992,1.974,2.862,2.909,...,3.873,1.054,0.780,0.843,1.182,0.947,0.623,0.659,0.601,0.720
2,1.463,2.570,3.189,3.980,4.387,3.726,3.408,2.366,2.610,2.540,...,1.356,1.074,0.746,0.541,0.570,0.612,0.764,2.123,1.846,1.820
3,1.748,1.924,2.060,2.029,2.021,1.864,2.754,1.979,2.286,2.111,...,3.179,3.394,4.462,3.008,2.127,1.018,1.445,1.840,1.820,1.928
4,2.320,2.828,2.691,2.520,2.190,1.975,1.950,1.514,1.985,1.665,...,2.069,1.433,1.139,1.328,2.120,2.451,1.974,2.975,2.447,2.465


## 2. Check the bundle

Look for `reload_check_passed: true` and finite test metrics. Segment 1, Segment 2, etc. are temporary names. Review the category profiles before assigning travel-style names.

In [3]:
print(json.dumps({key: manifest[key] for key in ["model_version", "k", "zero_policy", "mapping_status",
                                                                "test_metrics", "validation_uncertain_fraction",
                                                                "reload_check_passed", "registry_uploaded", "deployed"]}, indent=2))


{
  "model_version": "abfd88e893ca",
  "k": 5,
  "zero_policy": "keep_zero",
  "mapping_status": "provisional; review the cluster profiles",
  "test_metrics": {
    "silhouette": 0.14451545819379666,
    "davies_bouldin": 1.9913609603315208,
    "min_cluster_fraction": 0.042682926829268296,
    "cluster_counts": [
      281,
      86,
      278,
      35,
      140
    ]
  },
  "validation_uncertain_fraction": 0.0587515299877601,
  "reload_check_passed": true,
  "registry_uploaded": false,
  "deployed": false
}


## 3. Try a saved example

Use a known profile from the reload check. The same named inputs must return the saved cluster and version.

In [4]:
import joblib
import pandas as pd
fixture = json.loads((ROOT / "reports/golden-profiles.json").read_text())
bundle = joblib.load(ROOT / manifest["artifact"])
result = predict_profiles(bundle, pd.DataFrame(fixture["profiles"]))
assert result == fixture["expected"]
pd.DataFrame(result)


,cluster_id,segment,distance,uncertain,model_version
0,0,Segment 1,4.143480,False,abfd88e893ca
1,1,Segment 2,3.642955,False,abfd88e893ca
2,2,Segment 3,4.091836,False,abfd88e893ca
3,3,Segment 4,6.934135,True,abfd88e893ca
4,4,Segment 5,5.946169,False,abfd88e893ca


## Output

`models/artifacts/<version>/bundle.joblib`, `reports/bundle-manifest.json`, `cluster-profiles.csv`, and `golden-profiles.json`. Send the 03 selection and 04 summary for review. Next: tracking, API, and cloud readiness.